In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import pickle

def run(episodes, is_training=True, render=False):

    env = gym.make('Taxi-v3', render_mode='human' if render else None)

    if is_training:
        q = np.zeros((env.observation_space.n, env.action_space.n))  # 500 x 6
    else:
        with open('taxi.pkl', 'rb') as f:
            q = pickle.load(f)

    learning_rate_a = 0.9        # alpha - tasa de aprendizaje
    discount_factor_g = 0.9      # gamma - factor de descuento
    epsilon = 1.0                # 100% acciones aleatorias al inicio
    epsilon_decay_rate = 0.0001  # decae cada episodio
    rng = np.random.default_rng()

    rewards_per_episode = np.zeros(episodes)

    for i in range(episodes):
        state = env.reset()[0]
        terminated = False
        truncated = False
        rewards = 0

        while not terminated and not truncated:
            if is_training and rng.random() < epsilon:
                action = env.action_space.sample()  # 0=sur,1=norte,2=este,3=oeste,4=pickup,5=dropoff
            else:
                action = np.argmax(q[state, :])

            new_state, reward, terminated, truncated, _ = env.step(action)
            rewards += reward

            if is_training:
                q[state, action] = q[state, action] + learning_rate_a * (
                    reward + discount_factor_g * np.max(q[new_state, :]) - q[state, action]
                )

            state = new_state

        epsilon = max(epsilon - epsilon_decay_rate, 0)
        if epsilon == 0:
            learning_rate_a = 0.0001

        rewards_per_episode[i] = rewards

        if is_training and i % 1000 == 0:
            print(f"Episodio {i}/15000 - Epsilon: {epsilon:.4f} - Reward: {rewards}")

    env.close()

    sum_rewards = np.zeros(episodes)
    for t in range(episodes):
        sum_rewards[t] = np.sum(rewards_per_episode[max(0, t - 100):(t + 1)])

    plt.figure(figsize=(10, 5))
    plt.plot(sum_rewards)
    plt.title('Recompensas acumuladas - Taxi-v3')
    plt.xlabel('Episodios')
    plt.ylabel('Suma de recompensas (últimos 100)')
    plt.savefig('taxi.png')
    plt.show()

    if is_training:
        with open('taxi.pkl', 'wb') as f:
            pickle.dump(q, f)
        print("Modelo guardado en taxi.pkl")

In [ ]:
run(15000)

In [ ]:
run(10, is_training=False, render=True)